# Explore One Health indicators with fictional reef sites

**Goal:** load a small dataset, compare indicators with different units, and see how assumptions influence a weighted score.

**All sites and values are synthetic teaching examples.** They are not measurements from a real place or any mapping provider. The score is arbitrary and unvalidated; it is not a reef health, human health, or conservation-priority index. It must not guide real decisions.

This notebook uses Python's standard library and built-in notebook display tools. Run all cells in order.

## Read and connect

Read the abstract of [Sweet, Burian, and Bulling (2021)](https://pubmed.ncbi.nlm.nih.gov/33545133/) and the [Coral Reef Alliance introduction to reefs and human health](https://coral.org/blog/the-surprising-connections-between-coral-reefs-and-human-health/). Identify one proposed connection and the evidence needed to investigate it. The synthetic scores below do not test either reading or measure health outcomes.


In [1]:
import csv
import math
from pathlib import Path
from html import escape
from IPython.display import HTML, display

data_path = Path("data/synthetic-reef-sites.csv")
if not data_path.exists():
    raise FileNotFoundError("Open this notebook beside the data folder. In JupyterLite, refresh after the service worker starts and allow site storage.")
with data_path.open(encoding="utf-8", newline="") as handle:
    sites = [{key: value if key == "site" else float(value) for key, value in row.items()} for row in csv.DictReader(handle)]
print(f"Loaded {len(sites)} FICTIONAL reef sites.")
for site in sites:
    print(site)

Loaded 4 FICTIONAL reef sites.
{'site': 'Lagoon A', 'coral_cover_pct': 62.0, 'turbidity_ntu': 1.5, 'fish_species_observed': 42.0, 'community_dependence_pct': 75.0}
{'site': 'Lagoon B', 'coral_cover_pct': 28.0, 'turbidity_ntu': 6.0, 'fish_species_observed': 19.0, 'community_dependence_pct': 85.0}
{'site': 'Lagoon C', 'coral_cover_pct': 45.0, 'turbidity_ntu': 3.0, 'fish_species_observed': 31.0, 'community_dependence_pct': 40.0}
{'site': 'Lagoon D', 'coral_cover_pct': 72.0, 'turbidity_ntu': 0.8, 'fish_species_observed': 48.0, 'community_dependence_pct': 60.0}


## Understand the columns

- `coral_cover_pct`: synthetic live coral cover, percent.
- `turbidity_ntu`: synthetic turbidity, in nephelometric turbidity units.
- `fish_species_observed`: synthetic count of observed fish species; observation effort is assumed equal in this exercise.
- `community_dependence_pct`: fictional percentage representing dependence on reef-related activities. This is a teaching construct, not a measured or standardized social indicator.

The exercise treats lower coral cover and fish richness, higher turbidity, and higher dependence as increasing a toy score. These directions express assumptions, not an established scientific index.

In [2]:
# Fixed classroom reference scales; not ecological or health thresholds.
def bounded(value):
    return max(0.0, min(1.0, value))

def components(site):
    return {
        "coral": 1 - bounded(site["coral_cover_pct"] / 100),
        "water": bounded(site["turbidity_ntu"] / 10),
        "fish": 1 - bounded(site["fish_species_observed"] / 50),
        "dependence": bounded(site["community_dependence_pct"] / 100),
    }

weights = {"coral": 0.35, "water": 0.30, "fish": 0.20, "dependence": 0.15}

def score(site, selected_weights):
    if set(selected_weights) != set(components(site)):
        raise ValueError("Supply one weight for each component")
    if any(not math.isfinite(w) or w < 0 for w in selected_weights.values()) or not math.isclose(sum(selected_weights.values()), 1.0):
        raise ValueError("Weights must be finite, non-negative, and sum to 1")
    return 100 * sum(selected_weights[key] * value for key, value in components(site).items())

ranking = sorted(((site["site"], score(site, weights)) for site in sites), key=lambda item: item[1], reverse=True)
for name, value in ranking:
    print(f"{name}: {value:.1f} / 100 (synthetic teaching score)")

Lagoon B: 68.3 / 100 (synthetic teaching score)
Lagoon C: 41.9 / 100 (synthetic teaching score)
Lagoon A: 32.2 / 100 (synthetic teaching score)
Lagoon D: 22.0 / 100 (synthetic teaching score)


## Visualize the teaching scores

Bars compare the same unitless classroom score on a fixed 0-100 scale. They do not express probabilities, risk thresholds, or predicted outcomes.

In [3]:
bars = []
for name, value in ranking:
    bars.append(f'<div style="margin:12px 0"><span>{escape(name)}: {value:.1f}</span><div style="background:#e5eded;max-width:600px"><div style="height:20px;background:#087f8c;width:{value:.2f}%"></div></div></div>')
display(HTML('<div role="img" aria-label="Synthetic teaching scores on a zero to one hundred scale"><strong>Fictional sites | synthetic teaching scores (0-100)</strong>' + ''.join(bars) + '</div>'))

## Change the assumptions

Edit the alternative weights below, keep them non-negative and summing to 1, and rerun. A higher dependence weight need not imply that a community is unhealthy: in real work, indicator meaning and governance must be agreed with the people concerned.

In [4]:
alternative_weights = {"coral": 0.10, "water": 0.10, "fish": 0.10, "dependence": 0.70}
for site in sites:
    baseline = score(site, weights)
    alternative = score(site, alternative_weights)
    print(f"{site['site']}: baseline {baseline:.1f}; alternative {alternative:.1f}; change {alternative - baseline:+.1f}")

alternative_order = sorted(sites, key=lambda site: score(site, alternative_weights), reverse=True)
print("Alternative order:", " > ".join(site["site"] for site in alternative_order))

Lagoon A: baseline 32.2; alternative 59.4; change +27.2
Lagoon B: baseline 68.3; alternative 78.9; change +10.5
Lagoon C: baseline 41.9; alternative 40.3; change -1.6
Lagoon D: baseline 22.0; alternative 46.0; change +24.0
Alternative order: Lagoon B > Lagoon A > Lagoon D > Lagoon C


## Reflect before moving to real data

1. Which assumptions set the score's direction and scale? How would clipping affect extreme values?
2. What happens to the site ordering when community dependence receives more weight?
3. How could missing data, unequal survey effort, or mismatched observation dates distort a comparison?
4. What community permissions, local knowledge, field validation, and uncertainty estimates would be needed for an actual study?

The [WHO One Health overview](https://www.who.int/health-topics/one-health) provides broader context. This notebook demonstrates calculation and sensitivity, not causal inference or a validated assessment method.